# Desafio 01 – Ingestão e Padronização de Dados do IPCA-15 (IBGE / SIDRA)

Fonte: Tabela 1705 do SIDRA – IPCA15 - Variação mensal, acumulada no ano, 
acumulada em 12 meses e peso mensal, para o índice geral, grupos, subgrupos, 
itens e subitens de produtos e serviços (a partir de fev/2012).

Neste desafio vamos:

1. Consumir a API do SIDRA (tabela 1705) via HTTP (JSON);
2. Coletar a série mensal de jan/2019 a dez/2019 (Brasil, índice geral);
3. Padronizar o DataFrame (tipos, nomes de colunas);
4. Salvar dados brutos e tratados em `data/raw`, `data/processed` e `data/curated`.


In [2]:
# Importação de bibliotecas
from pathlib import Path
import requests
import json
import re
import pyarrow

import pandas as pd

In [3]:
# Definição de diretórios
BASE = Path.cwd().parent # sobe um nível na hierarquia de pastas
raw = BASE / "data" / "raw"
processed = BASE / "data" / "processed"
curated = BASE / "data" / "curated"


for f in [raw, processed, curated]:
    f.mkdir(parents=True, exist_ok=True)
    
BASE, raw, processed, curated


(WindowsPath('c:/workspace/data-engineering-challenge/01-data_ingestion'),
 WindowsPath('c:/workspace/data-engineering-challenge/01-data_ingestion/data/raw'),
 WindowsPath('c:/workspace/data-engineering-challenge/01-data_ingestion/data/processed'),
 WindowsPath('c:/workspace/data-engineering-challenge/01-data_ingestion/data/curated'))

In [4]:
url = "https://servicodados.ibge.gov.br/api/v3/agregados/1936/periodos/-6/variaveis/630|707|708|4032|4033|5956|5957|662|4036|4037?localidades=N1[all]&classificacao=12762[all]"

if "view=flat" not in url:
    separador = "&" if "?" in url else "?"
    url = url + f"{separador}view=flat"
else:
    url = url

url

req = requests.get(url)
data = req.json()
len(data)


raw_path = raw / "ibge_demografica_empresas.json"
with raw_path.open("w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

raw_path

WindowsPath('c:/workspace/data-engineering-challenge/01-data_ingestion/data/raw/ibge_demografica_empresas.json')

In [5]:
# Conversão para DataFrame
df_raw = pd.DataFrame(data)
print(df_raw.shape)
df_raw.head()

header = df_raw.iloc[0]
df = df_raw.iloc[1:].copy()
df.columns = header
df.reset_index(drop=True, inplace=True)

print(df.shape)
df.head()

(6541, 15)
(6540, 15)


,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Brasil (Código),Brasil,Ano (Código),Ano,Variável (Código),Variável,Classificação Nacional de Atividades Econômicas (CNAE 2.0) (Código),Classificação Nacional de Atividades Econômicas (CNAE 2.0),Faixas de pessoal ocupado (Código),Faixas de pessoal ocupado
0,1,Brasil,1020,Unidades,4481596,1,Brasil,2016,2016,630,Número de empresas,117897,Total,104029,Total
1,1,Brasil,1020,Unidades,33268,1,Brasil,2016,2016,630,Número de empresas,116830,"A Agricultura, pecuária, produção florestal, p...",104029,Total
2,1,Brasil,1020,Unidades,24909,1,Brasil,2016,2016,630,Número de empresas,116831,"01 Agricultura, pecuária e serviços relacionados",104029,Total
3,1,Brasil,1020,Unidades,6726,1,Brasil,2016,2016,630,Número de empresas,116866,02 Produção florestal,104029,Total
4,1,Brasil,1020,Unidades,1633,1,Brasil,2016,2016,630,Número de empresas,116873,03 Pesca e aquicultura,104029,Total


In [6]:
#Padronização dos nomes das colunas

def padroniza_string(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[áàâãä]", "a", s)
    s = re.sub(r"[éèêë]", "e", s)
    s = re.sub(r"[íìîï]", "i", s)
    s = re.sub(r"[óòôõö]", "o", s)
    s = re.sub(r"[úùûü]", "u", s)
    s = re.sub(r"ç", "c", s)
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s)
    s = s.strip("_")
    return s

df.columns = [padroniza_string(c) for c in df.columns]
df.head()

,nivel_territorial_codigo,nivel_territorial,unidade_de_medida_codigo,unidade_de_medida,valor,brasil_codigo,brasil,ano_codigo,ano,variavel_codigo,variavel,classificacao_nacional_de_atividades_economicas_cnae_2_0_codigo,classificacao_nacional_de_atividades_economicas_cnae_2_0,faixas_de_pessoal_ocupado_codigo,faixas_de_pessoal_ocupado
0,1,Brasil,1020,Unidades,4481596,1,Brasil,2016,2016,630,Número de empresas,117897,Total,104029,Total
1,1,Brasil,1020,Unidades,33268,1,Brasil,2016,2016,630,Número de empresas,116830,"A Agricultura, pecuária, produção florestal, p...",104029,Total
2,1,Brasil,1020,Unidades,24909,1,Brasil,2016,2016,630,Número de empresas,116831,"01 Agricultura, pecuária e serviços relacionados",104029,Total
3,1,Brasil,1020,Unidades,6726,1,Brasil,2016,2016,630,Número de empresas,116866,02 Produção florestal,104029,Total
4,1,Brasil,1020,Unidades,1633,1,Brasil,2016,2016,630,Número de empresas,116873,03 Pesca e aquicultura,104029,Total


In [50]:
# Salvar o arquivo processado
processed_path = processed / "ibge_demografica_empresas.parquet"

df_to_save = df.reset_index(drop=True).copy()

df_to_save.to_parquet(str(processed_path), engine="fastparquet", index=False)
processed_path

WindowsPath('c:/workspace/data-engineering-challenge/01-data_ingestion/data/processed/ibge_demografica_empresas.parquet')

Realizando o processo de refinamento

In [84]:
df.columns.tolist()

['nivel_territorial_codigo',
 'nivel_territorial',
 'unidade_de_medida_codigo',
 'unidade_de_medida',
 'valor',
 'brasil_codigo',
 'brasil',
 'ano_codigo',
 'ano',
 'variavel_codigo',
 'variavel',
 'classificacao_nacional_de_atividades_economicas_cnae_2_0_codigo',
 'classificacao_nacional_de_atividades_economicas_cnae_2_0',
 'faixas_de_pessoal_ocupado_codigo',
 'faixas_de_pessoal_ocupado']

In [85]:
cols_keep = [
    "ano_codigo",
    "variavel_codigo",
    "variavel",
    "classificacao_nacional_de_atividades_economicas_cnae_2_0_codigo",
    "classificacao_nacional_de_atividades_economicas_cnae_2_0",
    "faixas_de_pessoal_ocupado_codigo",
    "faixas_de_pessoal_ocupado",
    "valor"
]

df_curated = df[cols_keep].copy()
df_curated.head()

,ano_codigo,variavel_codigo,variavel,classificacao_nacional_de_atividades_economicas_cnae_2_0_codigo,classificacao_nacional_de_atividades_economicas_cnae_2_0,faixas_de_pessoal_ocupado_codigo,faixas_de_pessoal_ocupado,valor
0,2016,630,Número de empresas,117897,Total,104029,Total,4481596
1,2016,630,Número de empresas,116830,"A Agricultura, pecuária, produção florestal, p...",104029,Total,33268
2,2016,630,Número de empresas,116831,"01 Agricultura, pecuária e serviços relacionados",104029,Total,24909
3,2016,630,Número de empresas,116866,02 Produção florestal,104029,Total,6726
4,2016,630,Número de empresas,116873,03 Pesca e aquicultura,104029,Total,1633


In [86]:
df_curated = df_curated.rename(columns={
    "ano_codigo": "ano",
    "variavel_codigo": "variavel_codigo",
    "variavel": "variavel_nome",

    "classificacao_nacional_de_atividades_economicas_cnae_2_0_codigo": "cnae_codigo",
    "classificacao_nacional_de_atividades_economicas_cnae_2_0": "cnae_nome",

    "faixas_de_pessoal_ocupado_codigo": "faixa_ocupados_codigo",
    "faixas_de_pessoal_ocupado": "faixa_ocupados_nome",
})

In [87]:
df_curated.head()

,ano,variavel_codigo,variavel_nome,cnae_codigo,cnae_nome,faixa_ocupados_codigo,faixa_ocupados_nome,valor
0,2016,630,Número de empresas,117897,Total,104029,Total,4481596
1,2016,630,Número de empresas,116830,"A Agricultura, pecuária, produção florestal, p...",104029,Total,33268
2,2016,630,Número de empresas,116831,"01 Agricultura, pecuária e serviços relacionados",104029,Total,24909
3,2016,630,Número de empresas,116866,02 Produção florestal,104029,Total,6726
4,2016,630,Número de empresas,116873,03 Pesca e aquicultura,104029,Total,1633


In [88]:
#tipagem
df_curated["ano"] = df_curated["ano"].astype(int)
df_curated["variavel_codigo"] = df_curated["variavel_codigo"].astype(int)
df_curated["variavel_nome"] = df_curated["variavel_nome"].astype(str)
df_curated["cnae_codigo"] = df_curated["cnae_codigo"].astype(int)
df_curated["cnae_nome"] = df_curated["cnae_nome"].astype(str)
df_curated["faixa_ocupados_codigo"] = df_curated["faixa_ocupados_codigo"].astype(int)

df_curated["valor"] = pd.to_numeric(df_curated["valor"], errors="coerce")

df_curated.head()

,ano,variavel_codigo,variavel_nome,cnae_codigo,cnae_nome,faixa_ocupados_codigo,faixa_ocupados_nome,valor
0,2016,630,Número de empresas,117897,Total,104029,Total,4481596.0
1,2016,630,Número de empresas,116830,"A Agricultura, pecuária, produção florestal, p...",104029,Total,33268.0
2,2016,630,Número de empresas,116831,"01 Agricultura, pecuária e serviços relacionados",104029,Total,24909.0
3,2016,630,Número de empresas,116866,02 Produção florestal,104029,Total,6726.0
4,2016,630,Número de empresas,116873,03 Pesca e aquicultura,104029,Total,1633.0


In [89]:
# remoção de linhas totais

df_curated = df_curated[
    ~(df_curated["cnae_nome"].str.lower().eq("total"))
]

df_curated.head()

,ano,variavel_codigo,variavel_nome,cnae_codigo,cnae_nome,faixa_ocupados_codigo,faixa_ocupados_nome,valor
1,2016,630,Número de empresas,116830,"A Agricultura, pecuária, produção florestal, p...",104029,Total,33268.0
2,2016,630,Número de empresas,116831,"01 Agricultura, pecuária e serviços relacionados",104029,Total,24909.0
3,2016,630,Número de empresas,116866,02 Produção florestal,104029,Total,6726.0
4,2016,630,Número de empresas,116873,03 Pesca e aquicultura,104029,Total,1633.0
5,2016,630,Número de empresas,116880,B Indústrias extrativas,104029,Total,10295.0


In [90]:
# salvar o arquivo curated

curated_path = curated / "empresas_por_cnae_curated.parquet"
df_curated.to_parquet(curated_path, engine="fastparquet", index=False)
curated_path

WindowsPath('c:/workspace/data-engineering-challenge/01-data_ingestion/data/curated/empresas_por_cnae_curated.parquet')

### Explorando os dados e trazendo insights

In [91]:
print("Anos:", df_curated['ano'].unique())
print("CNAEs distintos:", df_curated['cnae_codigo'].nunique())
print("Faixas ocupados:", df_curated['faixa_ocupados_nome'].unique())
print("Variáveis:", df_curated['variavel_nome'].unique())


Anos: [2016 2017 2018 2019 2020 2021]
CNAEs distintos: 108
Faixas ocupados: ['Total']
Variáveis: ['Número de empresas' 'Pessoal ocupado total'
 'Pessoal ocupado assalariado'
 'Pessoal ocupado assalariado do sexo masculino das empresas'
 'Pessoal ocupado assalariado do sexo feminino das empresas'
 'Pessoal assalariado médio do sexo masculino das empresas'
 'Pessoal assalariado médio do sexo feminino das empresas'
 'Salários e outras remunerações'
 'Salários e outras remunerações dos empregados do sexo masculino das empresas'
 'Salários e outras remunerações dos empregados do sexo feminino das empresas']


In [92]:
print("Serviços de TI:", df_curated[df_curated['cnae_nome'].str.contains('62 Atividades dos serviços de tecnologia da informação', case=False)])

Serviços de TI:        ano  variavel_codigo  \
65    2016              630   
174   2016              707   
283   2016              708   
392   2016             4032   
501   2016             4033   
610   2016             5956   
719   2016             5957   
828   2016              662   
937   2016             4036   
1046  2016             4037   
1155  2017              630   
1264  2017              707   
1373  2017              708   
1482  2017             4032   
1591  2017             4033   
1700  2017             5956   
1809  2017             5957   
1918  2017              662   
2027  2017             4036   
2136  2017             4037   
2245  2018              630   
2354  2018              707   
2463  2018              708   
2572  2018             4032   
2681  2018             4033   
2790  2018             5956   
2899  2018             5957   
3008  2018              662   
3117  2018             4036   
3226  2018             4037   
3335  2019             

In [93]:
# Analise por genero e salario e tipo de cnae:

# variáveis de salário por gênero
var_nome = df_curated["variavel_nome"].isin([
    "Salários e outras remunerações dos empregados do sexo masculino das empresas",
    "Salários e outras remunerações dos empregados do sexo feminino das empresas",
])

# CNAE 62 - serviços de TI
cnae_ti = df_curated["cnae_nome"].str.contains(
    "62 Atividades dos serviços de tecnologia da informação",
    case=False,
    na=False
)

# aplica os dois filtros ao mesmo tempo
df_genero_salario = (
    df_curated[var_nome & cnae_ti]
    .groupby(["ano", "cnae_nome", "variavel_nome"])["valor"]
    .sum()
    .unstack()
)

# renomeia colunas
df_genero_salario = df_genero_salario.rename(columns={
    "Salários e outras remunerações dos empregados do sexo masculino das empresas": "salario_masc",
    "Salários e outras remunerações dos empregados do sexo feminino das empresas": "salario_fem",
})

df_genero_salario.head()

,variavel_nome,salario_fem,salario_masc
ano,cnae_nome,,
2016,62 Atividades dos serviços de tecnologia da informação,6418451.0,17772319.0
2017,62 Atividades dos serviços de tecnologia da informação,6844823.0,18944554.0
2018,62 Atividades dos serviços de tecnologia da informação,7225582.0,19971359.0
2019,62 Atividades dos serviços de tecnologia da informação,8109923.0,22139115.0
2020,62 Atividades dos serviços de tecnologia da informação,8919627.0,24192354.0


In [95]:
# Dados agregados por genero (empregados) e cnae

#variavel de genero (empregados)
var_genero = df_curated.variavel_nome.isin([
        "Pessoal ocupado assalariado do sexo masculino das empresas",
        "Pessoal ocupado assalariado do sexo feminino das empresas"
    ])

# CNAE 62 - serviços de TI
cnae_ti = df_curated["cnae_nome"].str.contains(
    "62 Atividades dos serviços de tecnologia da informação",
    case=False,
    na=False
)

df_genero = (
    df_curated[var_genero & cnae_ti]
    .groupby(["ano", "cnae_nome", "variavel_nome"])["valor"]
    .sum()
    .unstack()
)

df_genero = df_genero.rename(columns={
    "Pessoal ocupado assalariado do sexo masculino das empresas": "empregados_masc",
    "Pessoal ocupado assalariado do sexo feminino das empresas": "empregadas_fem",
})

df_genero.head()

,variavel_nome,empregadas_fem,empregados_masc
ano,cnae_nome,,
2016,62 Atividades dos serviços de tecnologia da informação,124530.0,244191.0
2017,62 Atividades dos serviços de tecnologia da informação,125111.0,246316.0
2018,62 Atividades dos serviços de tecnologia da informação,132027.0,259947.0
2019,62 Atividades dos serviços de tecnologia da informação,145117.0,280361.0
2020,62 Atividades dos serviços de tecnologia da informação,151035.0,291397.0


In [96]:
# Confirmação de ordenação dos índices
df_genero_salario = df_genero_salario.sort_index()
df_genero = df_genero.sort_index()

In [97]:
# Cálculo do salário médio por gênero
df_avg_salario = pd.DataFrame(index=df_genero.index)

df_avg_salario["sal_medio_masc"] = (
    df_genero_salario["salario_masc"] / df_genero["empregados_masc"]
)

df_avg_salario["sal_medio_fem"] = (
    df_genero_salario["salario_fem"] / df_genero["empregadas_fem"]
)


df_avg_salario.head(50)

,,sal_medio_masc,sal_medio_fem
ano,cnae_nome,,
2016,62 Atividades dos serviços de tecnologia da informação,72.780401,51.541404
2017,62 Atividades dos serviços de tecnologia da informação,76.911585,54.710002
2018,62 Atividades dos serviços de tecnologia da informação,76.828580,54.728063
2019,62 Atividades dos serviços de tecnologia da informação,78.966458,55.885410
2020,62 Atividades dos serviços de tecnologia da informação,83.021973,59.056689
2021,62 Atividades dos serviços de tecnologia da informação,88.621355,62.129471
